In [ ]:
# =========================
# 1. INSTALL & ENVIRONMENT CONFIG
# =========================
!pip -q install ultralytics==8.4.50

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["YOLO_VERBOSE"] = "True"
os.environ["ULTRALYTICS_VERBOSE"] = "True"
os.environ["PYTHONWARNINGS"] = "ignore"

print("Ultralytics installed and environment configured.")

In [ ]:
# =========================
# 2. GLOBAL CONFIG (KAGGLE VERSION)
# =========================
from pathlib import Path
import os, shutil, random, zipfile, json, yaml, time, gc, hashlib
from collections import Counter, defaultdict

# Đường dẫn đến các Dataset đã add vào Notebook (Hãy kiểm tra lại tên thư mục chính xác ở góc phải màn hình)
KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets/dophucvuha/yolo-ppe-merged-dataset")
KAGGLE_INPUT_DIR_2 = Path("/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow")

KAGGLE_INPUT_DIRS = [d for d in [KAGGLE_INPUT_DIR, KAGGLE_INPUT_DIR_2] if d.exists()]

# Lưu kết quả ở thư mục làm việc của Kaggle
PROJECT_ROOT = Path("/kaggle/working/Final_ComputerVision")
WORK_DIR = Path("/kaggle/working/ppe_merge_train_work")

RAW_DIR = WORK_DIR / "raw_datasets"
MERGED_DATASET_DIR = WORK_DIR / "merged_ppe_dataset"
RUNS_DIR = WORK_DIR / "runs"

FINAL_MODEL_DIR = PROJECT_ROOT / "final_models"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Tên gợi ý file ZIP (Lúc này đã được Kaggle giải nén thành các thư mục con)
ZIP_NAME_HINTS = [
    "BAC_HIEN_CONSTRUCTION_SAFETY_2024.v1i.yolov8",
    "Pelanggaran APD- 5 kelas.v1i.yolov8",
]

TRAIN_PROFILE = "balanced"  # fast / balanced / strong
MAP_PPE_TO_VEST = False     

BASE_MODEL = "yolo11s.pt"
SEED = 42
random.seed(SEED)

PROFILES = {
    "fast": {
        "max_train_images_total": 12000, "max_val_images_total": 1800, "max_test_images_total": 1800,
        "stage1_epochs": 40, "stage1_patience": 8, "batch_stage1": 16, "batch_stage2": 8, # Nâng batch size vì GPU Kaggle mạnh hơn
    },
    "balanced": {
        "max_train_images_total": 16000, "max_val_images_total": 2400, "max_test_images_total": 2400,
        "stage1_epochs": 50, "stage1_patience": 10, "batch_stage1": 16, "batch_stage2": 8,
    },
    "strong": {
        "max_train_images_total": 22000, "max_val_images_total": 3000, "max_test_images_total": 3000,
        "stage1_epochs": 60, "stage1_patience": 12, "batch_stage1": 16, "batch_stage2": 8,
    },
}
CFG = PROFILES[TRAIN_PROFILE]

if MAP_PPE_TO_VEST:
    TARGET_CLASSES = ["person", "helmet", "no_helmet", "vest", "no_vest", "no_glove"]
else:
    TARGET_CLASSES = ["person", "helmet", "no_helmet", "no_vest", "no_glove"]

WORKERS = 1

CACHE = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK_DIR:", WORK_DIR)
print("TARGET_CLASSES:", TARGET_CLASSES)

In [ ]:

# =========================
# 3. UTILS
# =========================
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import hashlib
import yaml
import shutil
import random
import zipfile

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def read_yaml(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def write_yaml(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

def parse_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
    if isinstance(names, list):
        return names
    return []

def normalize_name(s):
    return (
        str(s).strip().lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(".", "")
        .replace(",", "")
    )

def file_md5(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def collect_images(path):
    path = Path(path)
    if not path.exists():
        return []
    if path.is_file():
        with open(path, "r", encoding="utf-8") as f:
            return [Path(line.strip()) for line in f if line.strip()]
    return [p for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTS]

def resolve_split_path(data_yaml_path, split_value):
    data_yaml_path = Path(data_yaml_path)
    data = read_yaml(data_yaml_path)
    root = Path(data.get("path", data_yaml_path.parent))
    if not root.is_absolute():
        root = data_yaml_path.parent / root
    split_path = Path(str(split_value))
    if not split_path.is_absolute():
        split_path = root / split_path
    return split_path.resolve()

def image_to_label_path(img_path):
    img_path = Path(img_path)
    parts = list(img_path.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() == "images":
            parts[i] = "labels"
            return Path(*parts).with_suffix(".txt")
    return img_path.parent.parent / "labels" / (img_path.stem + ".txt")

def read_yolo_labels(label_path, names_len=None):
    labels = []
    label_path = Path(label_path)
    if not label_path.exists():
        return labels, 0
    invalid = 0
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return labels, invalid
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            invalid += 1
            continue
        try:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
        except Exception:
            invalid += 1
            continue
        if names_len is not None and not (0 <= cls < names_len):
            invalid += 1
            continue
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            invalid += 1
            continue
        labels.append((cls, x, y, w, h, parts[1:]))
    return labels, invalid

def audit_dataset(data_yaml_path, title="DATASET"):
    data_yaml_path = Path(data_yaml_path)
    data = read_yaml(data_yaml_path)
    names = parse_names(data.get("names", []))
    print("\n" + "=" * 90)
    print(title)
    print("YAML:", data_yaml_path)
    print("Names:", names)
    report = {}
    for split in ["train", "val", "valid", "test"]:
        if split not in data or data.get(split) in [None, ""]:
            continue
        split_path = resolve_split_path(data_yaml_path, data[split])
        images = collect_images(split_path)
        counter = Counter()
        bg = 0
        missing = 0
        invalid = 0
        for img in images:
            lab = image_to_label_path(img)
            labels, inv = read_yolo_labels(lab, names_len=len(names))
            invalid += inv
            if not lab.exists():
                missing += 1
            if len(labels) == 0:
                bg += 1
            for cls, *_ in labels:
                counter[names[cls]] += 1
        report[split] = {
            "images": len(images),
            "backgrounds": bg,
            "missing_labels": missing,
            "invalid_lines": invalid,
            "instances": dict(counter),
        }
        print(split, report[split])
    return report


In [ ]:

# =========================
# 5. CLASS MAPPING
# =========================
CLASS_ALIASES = {
    "person": ["person", "people", "worker", "workers", "human"],
    "helmet": [
        "helmet", "hardhat", "hard_hat", "hard-hat",
        "safety helmet", "safety_helmet", "safety-helmet",
        "with helmet", "with_helmet", "with-helmet",
    ],
    "no_helmet": [
        "non helmet", "non_helmet", "non-helmet", "nonhelmet",
        "no helmet", "no_helmet", "no-helmet", "nohelmet",
        "without helmet", "without_helmet", "without-helmet",
        "no hardhat", "no_hardhat", "no-hardhat",
    ],
    "vest": [
        "vest", "safety vest", "safety_vest", "safety-vest",
        "reflective vest", "reflective_vest", "reflective-vest",
        "ppe",
    ],
    "no_vest": [
        "no vest", "no_vest", "no-vest", "novest",
        "without vest", "without_vest", "without-vest",
        "no safety vest", "no_safety_vest", "no-safety-vest",
        "no reflective vest", "no_reflective_vest", "no-reflective-vest",
    ],
    "no_glove": [
        "no glove", "no_glove", "no-glove", "noglove",
        "no gloves", "no_gloves", "no-gloves",
        "without glove", "without_glove", "without-glove",
        "without gloves", "without_gloves", "without-gloves",
    ],
}

IGNORE_SOURCE_CLASSES = [
    "ppe", "mask", "no mask", "no_mask", "no-mask",
    "glove", "gloves", "vehicle", "machinery", "cone", "safety cone",
    "boots", "boot",
]

def build_optional_mapping(source_names, target_classes=TARGET_CLASSES, aliases=CLASS_ALIASES):
    mapping = {}
    unmapped = []
    ignored = []

    alias_lookup = {}
    for target_name in target_classes:
        alias_set = {normalize_name(target_name)}
        for a in aliases.get(target_name, []):
            if normalize_name(a) == "ppe" and not MAP_PPE_TO_VEST:
                continue
            alias_set.add(normalize_name(a))
        alias_lookup[target_name] = alias_set

    ignored_set = set(normalize_name(x) for x in IGNORE_SOURCE_CLASSES)
    if MAP_PPE_TO_VEST:
        ignored_set.discard("ppe")

    for src_id, src_name in enumerate(source_names):
        norm_src = normalize_name(src_name)
        matched_target = None

        for target_name, alias_set in alias_lookup.items():
            if norm_src in alias_set:
                matched_target = target_name
                break

        if matched_target is not None:
            target_id = target_classes.index(matched_target)
            mapping[src_id] = target_id
        else:
            if norm_src in ignored_set:
                ignored.append(src_name)
            else:
                unmapped.append(src_name)

    print("Source names:", source_names)
    print("Mapping source_id -> target_id:", mapping)
    print("Mapping readable:")
    for src_id, target_id in mapping.items():
        print(f"  {src_id}: {source_names[src_id]} -> {target_classes[target_id]}")

    if unmapped:
        print("Unmapped classes:", unmapped)
    if ignored:
        print("Ignored classes:", ignored)

    return mapping, unmapped, ignored


In [ ]:
# =========================
# 4. PROCESS KAGGLE INPUT DATASETS
# =========================
from pathlib import Path
import shutil

DATASET_SOURCES = []

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Lấy danh sách tất cả các thư mục con trong từng Input dataset đã gắn
subdirs = []
for input_dir in KAGGLE_INPUT_DIRS:
    children = [p for p in input_dir.iterdir() if p.is_dir()]
    if not children:
        # Trường hợp dataset không tạo thư mục gốc bọc ngoài
        children = [input_dir]
    subdirs.extend(children)

print("Found directories in input:", subdirs)

for idx, current_dir in enumerate(subdirs):
    dataset_id = f"ds{idx}_{current_dir.name[:45].replace(' ', '_').replace('-', '_')}"
    
    # Tìm file yaml trong thư mục này
    yaml_files = list(current_dir.rglob("data.yaml")) + list(current_dir.rglob("*.yaml"))
    yaml_files = sorted(set(yaml_files))
    
    if not yaml_files:
        continue
        
    original_yaml = yaml_files[0]
    data = read_yaml(original_yaml)
    names = parse_names(data.get("names", []))
    
    dataset_root = original_yaml.parent
    
    # Kiểm tra tính hợp lệ của cấu trúc YOLO
    if not (dataset_root / "train" / "images").exists():
        continue
        
    valid_key = "valid" if (dataset_root / "valid" / "images").exists() else "val"
    test_value = "test/images" if (dataset_root / "test" / "images").exists() else f"{valid_key}/images"
    
    # Tạo file YAML mới lưu vào thư mục ghi được (WORK_DIR) để YOLO không bị lỗi quyền ghi (Permission Denied)
    fixed_yaml = RAW_DIR / f"{dataset_id}_fixed.yaml"
    fixed = {
        "path": str(dataset_root.resolve()),
        "train": "train/images",
        "val": f"{valid_key}/images",
        "test": test_value,
        "nc": len(names),
        "names": names,
    }
    write_yaml(fixed_yaml, fixed)
    
    print(f"\n--- Processed: {dataset_id} ---")
    print("Root:", dataset_root)
    print("Classes:", names)
    
    audit_dataset(fixed_yaml, title=f"RAW AUDIT {dataset_id}")
    
    DATASET_SOURCES.append({
        "id": dataset_id,
        "yaml": fixed_yaml,
        "root": dataset_root,
        "names": names,
    })

assert len(DATASET_SOURCES) >= 2, "Không tìm đủ cấu trúc của cả 2 datasets, vui lòng kiểm tra lại cấu trúc thư mục input!"

In [ ]:

# =========================
# 6. MERGE + REMAP DATASETS SAFELY
# =========================
if MERGED_DATASET_DIR.exists():
    shutil.rmtree(MERGED_DATASET_DIR)

for split in ["train", "val", "test"]:
    (MERGED_DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

MAX_TRAIN_TOTAL = CFG["max_train_images_total"]
MAX_VAL_TOTAL = CFG["max_val_images_total"]
MAX_TEST_TOTAL = CFG["max_test_images_total"]

PRIORITY_CLASS_NAMES = ["no_helmet", "no_vest", "no_glove"]
if "vest" in TARGET_CLASSES:
    PRIORITY_CLASS_NAMES.append("vest")

PRIORITY_CLASS_IDS = {TARGET_CLASSES.index(cls_name) for cls_name in PRIORITY_CLASS_NAMES if cls_name in TARGET_CLASSES}

print("TARGET_CLASSES:", TARGET_CLASSES)
print("PRIORITY_CLASS_IDS:", PRIORITY_CLASS_IDS)
print("MAX_TRAIN_TOTAL:", MAX_TRAIN_TOTAL)

seen_hashes = set()
all_items_by_split = defaultdict(list)
merge_stats = defaultdict(lambda: {
    "images": 0,
    "backgrounds": 0,
    "instances": Counter(),
    "skipped_no_target": 0,
    "duplicates": 0,
    "invalid_lines": 0,
})

def remap_label_lines(label_path, source_names, mapping):
    labels, invalid = read_yolo_labels(label_path, names_len=len(source_names))
    new_lines = []
    original_had_labels = len(labels) > 0
    class_ids = []
    for cls, x, y, w, h, rest in labels:
        if cls not in mapping:
            continue
        new_cls = mapping[cls]
        new_lines.append(" ".join([str(new_cls)] + rest))
        class_ids.append(new_cls)
    return new_lines, original_had_labels, class_ids, invalid

def split_name_from_key(key):
    if key == "train":
        return "train"
    if key in ["val", "valid"]:
        return "val"
    if key == "test":
        return "test"
    return None

for src in DATASET_SOURCES:
    data_yaml = Path(src["yaml"])
    data = read_yaml(data_yaml)
    source_names = src["names"]

    print("\n" + "=" * 90)
    print("PROCESS DATASET:", src["id"])
    print("YAML:", data_yaml)
    print("Source names:", source_names)

    mapping, unmapped, ignored = build_optional_mapping(source_names)
    if not mapping:
        print("SKIP dataset because no mapped classes:", src["id"])
        continue

    used_out_splits = set()
    for split_key in ["train", "val", "valid", "test"]:
        if split_key not in data or data.get(split_key) in [None, ""]:
            continue
        out_split = split_name_from_key(split_key)
        if out_split is None or out_split in used_out_splits:
            continue
        used_out_splits.add(out_split)

        split_path = resolve_split_path(data_yaml, data[split_key])
        images = collect_images(split_path)
        print(f"Scanning {src['id']} {split_key} -> {out_split}: {len(images)} images")

        for img_path in tqdm(images, desc=f"Scan {src['id']} {out_split}"):
            img_path = Path(img_path)
            lab = image_to_label_path(img_path)
            new_lines, original_had_labels, class_ids, invalid = remap_label_lines(lab, source_names, mapping)
            merge_stats[out_split]["invalid_lines"] += invalid

            if original_had_labels and len(new_lines) == 0:
                merge_stats[out_split]["skipped_no_target"] += 1
                continue

            priority = sum(1 for c in class_ids if c in PRIORITY_CLASS_IDS)
            item = {"src_id": src["id"], "img": img_path, "lines": new_lines, "class_ids": class_ids, "priority": priority}
            all_items_by_split[out_split].append(item)

def sample_items(items, max_n, split):
    if max_n is None or len(items) <= max_n:
        return items
    if split == "train":
        items_sorted = sorted(items, key=lambda x: (x["priority"], random.random()), reverse=True)
        selected = items_sorted[:max_n]
        random.shuffle(selected)
        return selected
    random.shuffle(items)
    return items[:max_n]

selected_by_split = {
    "train": sample_items(all_items_by_split["train"], MAX_TRAIN_TOTAL, "train"),
    "val": sample_items(all_items_by_split["val"], MAX_VAL_TOTAL, "val"),
    "test": sample_items(all_items_by_split["test"], MAX_TEST_TOTAL, "test"),
}

print("\nSelected item counts:", {k: len(v) for k, v in selected_by_split.items()})

# Oversampling cho các lớp hiếm (chỉ áp dụng cho train, giữ val/test nguyên bản để eval trung thực)
OVERSAMPLE_CLASS_WEIGHTS = {"no_vest": 3, "no_glove": 6}

for split, items in selected_by_split.items():
    out_img_dir = MERGED_DATASET_DIR / split / "images"
    out_lab_dir = MERGED_DATASET_DIR / split / "labels"

    for item in tqdm(items, desc=f"Copy {split}"):
        img_path = Path(item["img"])
        h = file_md5(img_path)
        if h in seen_hashes:
            merge_stats[split]["duplicates"] += 1
            continue
        seen_hashes.add(h)

        safe_src = "".join(ch if ch.isalnum() else "_" for ch in item["src_id"])[:45]
        base_name = f"{safe_src}_{h[:10]}"

        repeat = 1
        if split == "train" and item["class_ids"]:
            repeat = max(OVERSAMPLE_CLASS_WEIGHTS.get(TARGET_CLASSES[c], 1) for c in item["class_ids"])

        for r in range(repeat):
            suffix = "" if r == 0 else f"_os{r}"
            new_name = f"{base_name}{suffix}{img_path.suffix.lower()}"
            dst_img = out_img_dir / new_name
            dst_lab = out_lab_dir / (Path(new_name).stem + ".txt")
            shutil.copy2(img_path, dst_img)
            dst_lab.write_text("\n".join(item["lines"]), encoding="utf-8")

            merge_stats[split]["images"] += 1
            if len(item["lines"]) == 0:
                merge_stats[split]["backgrounds"] += 1
            for c in item["class_ids"]:
                merge_stats[split]["instances"][TARGET_CLASSES[c]] += 1

        if repeat > 1:
            merge_stats[split]["oversampled_extra_copies"] = merge_stats[split].get("oversampled_extra_copies", 0) + (repeat - 1)

MERGED_DATA_YAML = MERGED_DATASET_DIR / "data.yaml"
write_yaml(MERGED_DATA_YAML, {
    "path": str(MERGED_DATASET_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images" if len(collect_images(MERGED_DATASET_DIR / "test" / "images")) > 0 else "val/images",
    "nc": len(TARGET_CLASSES),
    "names": TARGET_CLASSES,
})

print("\nMERGED_DATA_YAML:", MERGED_DATA_YAML)
print(MERGED_DATA_YAML.read_text(encoding="utf-8"))

print("\nMERGE STATS:")
for split, st in merge_stats.items():
    printable = dict(st)
    printable["instances"] = dict(printable["instances"])
    print(split, printable)

merged_report = audit_dataset(MERGED_DATA_YAML, title="MERGED DATASET AUDIT")

train_instances = merged_report.get("train", {}).get("instances", {})
print("\nTRAIN CLASS COUNTS:")
for cls in TARGET_CLASSES:
    print(cls, train_instances.get(cls, 0))

missing = [cls for cls in TARGET_CLASSES if train_instances.get(cls, 0) == 0]
if missing:
    print("\nWARNING: These target classes have 0 training boxes:", missing)
    print("Update CLASS_ALIASES or MAP_PPE_TO_VEST, then rerun Cell 5-6.")
else:
    print("\nAll target classes have training boxes. Ready to train.")


In [ ]:

# =========================
# 6.9 REDUCE TRAINING LAG
# =========================
import os
import gc
import torch

os.environ["WANDB_DISABLED"] = "true"
os.environ["YOLO_VERBOSE"] = "True"
os.environ["ULTRALYTICS_VERBOSE"] = "True"
os.environ["PYTHONWARNINGS"] = "ignore"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# =========================
# 7. TRAIN STAGE 1 - STABLE, LOW LAG, 40/50/60 EPOCHS
# =========================
import gc
import time
import torch
from pathlib import Path
from ultralytics import YOLO

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = 0 if torch.cuda.is_available() else "cpu"

STAGE1_NAME = f"ppe_merge_stage1_stable_{TRAIN_PROFILE}_{time.strftime('%Y%m%d_%H%M%S')}"
STAGE1_EPOCHS = CFG["stage1_epochs"]

STAGE1_CONFIG = dict(
    epochs=STAGE1_EPOCHS,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.0007,
    lrf=0.01,
    weight_decay=0.0005,
    patience=CFG["stage1_patience"],
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.08,
    box=7.5,
    cls=0.70,
    dfl=1.5,
    mosaic=0.30,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=8,
)

print("========== TRAIN STAGE 1 - STABLE ==========")
print("TRAIN_PROFILE:", TRAIN_PROFILE)
print("BASE_MODEL:", BASE_MODEL)
print("MERGED_DATA_YAML:", MERGED_DATA_YAML)
print("RUNS_DIR:", RUNS_DIR)
print("EPOCHS:", STAGE1_EPOCHS)
print("CONFIG:", STAGE1_CONFIG)

model = YOLO(BASE_MODEL)

results1 = model.train(
    data=str(MERGED_DATA_YAML),
    project=str(RUNS_DIR),
    name=STAGE1_NAME,
    exist_ok=True,
    device=device,
    seed=SEED,
    batch=CFG["batch_stage1"],
    workers=WORKERS,
    cache=CACHE,
    pretrained=True,
    amp=True,
    cos_lr=True,
    plots=True,
    verbose=True,
    save_period=1,
    val=True,
    hsv_h=0.008,
    hsv_s=0.20,
    hsv_v=0.14,
    degrees=1.0,
    translate=0.05,
    scale=0.25,
    shear=0.0,
    perspective=0.0,
    fliplr=0.50,
    flipud=0.0,
    **STAGE1_CONFIG,
)

STAGE1_DIR = Path(results1.save_dir)
STAGE1_BEST = STAGE1_DIR / "weights" / "best.pt"
STAGE1_LAST = STAGE1_DIR / "weights" / "last.pt"

if not STAGE1_BEST.exists() and STAGE1_LAST.exists():
    STAGE1_BEST = STAGE1_LAST

print("\n========== STAGE 1 DONE ==========")
print("STAGE1_DIR:", STAGE1_DIR)
print("STAGE1_BEST:", STAGE1_BEST)
print("STAGE1_LAST:", STAGE1_LAST)


In [ ]:

# =========================
# 7.1 VALIDATE STAGE 1 ONCE
# =========================
from ultralytics import YOLO
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_val = YOLO(str(STAGE1_BEST))

metrics_stage1 = model_val.val(
    data=str(MERGED_DATA_YAML),
    imgsz=640,
    batch=CFG["batch_stage1"],
    device=device,
    plots=True,
    verbose=True,
)

box = metrics_stage1.box
stage1_eval = {
    "name": "stage1",
    "path": str(STAGE1_BEST),
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
}
print("STAGE 1 EVAL:", stage1_eval)


In [ ]:

# =========================
# 8. OPTIONAL STAGE 2 REFINE 768
# =========================
RUN_STAGE2 = True

if RUN_STAGE2:
    import gc
    import time
    import torch
    from pathlib import Path
    from ultralytics import YOLO

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    STAGE2_NAME = f"ppe_merge_refine768_{TRAIN_PROFILE}_{time.strftime('%Y%m%d_%H%M%S')}"
    STAGE2_CONFIG = dict(
        epochs=8,
        imgsz=768,
        optimizer="AdamW",
        lr0=0.00005,
        lrf=0.01,
        weight_decay=0.0005,
        patience=4,
        warmup_epochs=1.0,
        box=7.5,
        cls=0.65,
        dfl=1.5,
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,
        close_mosaic=1,
    )

    print("========== TRAIN STAGE 2 REFINE 768 ==========")
    print("CONFIG:", STAGE2_CONFIG)

    model2 = YOLO(str(STAGE1_BEST))
    results2 = model2.train(
        data=str(MERGED_DATA_YAML),
        project=str(RUNS_DIR),
        name=STAGE2_NAME,
        exist_ok=True,
        device=device,
        seed=SEED,
        batch=CFG["batch_stage2"],
        workers=WORKERS,
        cache=CACHE,
        pretrained=True,
        amp=True,
        cos_lr=True,
        plots=True,
        verbose=True,
        save_period=1,
        val=True,
        hsv_h=0.004,
        hsv_s=0.10,
        hsv_v=0.08,
        degrees=0.5,
        translate=0.03,
        scale=0.12,
        shear=0.0,
        perspective=0.0,
        fliplr=0.50,
        flipud=0.0,
        **STAGE2_CONFIG,
    )

    STAGE2_DIR = Path(results2.save_dir)
    STAGE2_BEST = STAGE2_DIR / "weights" / "best.pt"
    STAGE2_LAST = STAGE2_DIR / "weights" / "last.pt"
    if not STAGE2_BEST.exists() and STAGE2_LAST.exists():
        STAGE2_BEST = STAGE2_LAST
else:
    STAGE2_DIR = STAGE1_DIR
    STAGE2_BEST = STAGE1_BEST
    STAGE2_LAST = STAGE1_LAST

print("STAGE2_DIR:", STAGE2_DIR)
print("STAGE2_BEST:", STAGE2_BEST)
print("STAGE2_LAST:", STAGE2_LAST)


In [ ]:

# =========================
# 8.1 VALIDATE STAGE 2 ONCE
# =========================
from ultralytics import YOLO
import gc
import torch

if RUN_STAGE2:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_val2 = YOLO(str(STAGE2_BEST))
    metrics_stage2 = model_val2.val(
        data=str(MERGED_DATA_YAML),
        imgsz=768,
        batch=CFG["batch_stage2"],
        device=device,
        plots=True,
        verbose=True,
    )

    box = metrics_stage2.box
    stage2_eval = {
        "name": "stage2",
        "path": str(STAGE2_BEST),
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }
    print("STAGE 2 EVAL:", stage2_eval)
else:
    stage2_eval = None
    print("Stage 2 skipped.")


In [ ]:

# =========================
# 9. SELECT BEST + SAVE FINAL MODEL TO DRIVE
# =========================
from pathlib import Path
import shutil
import time

evals = []
if "stage1_eval" in globals():
    evals.append(stage1_eval)
if "stage2_eval" in globals() and stage2_eval is not None:
    evals.append(stage2_eval)

assert evals, "Không có kết quả eval để chọn model."

best_eval = sorted(evals, key=lambda x: (x["mAP50_95"], x["mAP50"]), reverse=True)[0]

SELECTED_MODEL = Path(best_eval["path"])
SELECTED_NAME = best_eval["name"]

FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = FINAL_MODEL_DIR / "ppe_merged_best.pt"
ALIAS_MODEL_PATH = FINAL_MODEL_DIR / "ppe_safety_best.pt"
VERSIONED_MODEL_PATH = FINAL_MODEL_DIR / f"ppe_merged_{SELECTED_NAME}_{time.strftime('%Y%m%d_%H%M%S')}.pt"

shutil.copy2(SELECTED_MODEL, FINAL_MODEL_PATH)
shutil.copy2(SELECTED_MODEL, ALIAS_MODEL_PATH)
shutil.copy2(SELECTED_MODEL, VERSIONED_MODEL_PATH)

print("BEST EVAL:", best_eval)
print("SELECTED_MODEL:", SELECTED_MODEL)
print("Saved final model to:", FINAL_MODEL_PATH)
print("Saved alias model to:", ALIAS_MODEL_PATH)
print("Saved versioned model to:", VERSIONED_MODEL_PATH)


In [ ]:

# =========================
# 10. QUICK PREDICT TEST IMAGES
# =========================
from ultralytics import YOLO
from pathlib import Path
import torch

TEST_IMAGE_DIR = PROJECT_ROOT / "test_images"
PREDICT_DIR = PROJECT_ROOT / "predictions"
TEST_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREDICT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = FINAL_MODEL_DIR / "ppe_merged_best.pt"

print("FINAL_MODEL_PATH:", FINAL_MODEL_PATH)
print("TEST_IMAGE_DIR:", TEST_IMAGE_DIR)

if any(TEST_IMAGE_DIR.glob("*")):
    model_pred = YOLO(str(FINAL_MODEL_PATH))
    pred = model_pred.predict(
        source=str(TEST_IMAGE_DIR),
        imgsz=768,
        conf=0.35,
        iou=0.5,
        save=True,
        project=str(PREDICT_DIR),
        name="predict_test_images",
        exist_ok=True,
        device=0 if torch.cuda.is_available() else "cpu",
    )
    print("Predictions saved to:", PREDICT_DIR / "predict_test_images")
else:
    print("Chưa có ảnh test. Hãy upload ảnh vào:", TEST_IMAGE_DIR)
